In [2]:
import pandas as pd
import numpy as np
from dataclasses import dataclass
import dataclasses
import json
from kafka import KafkaProducer

In [3]:
url = 'https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2025-10.parquet'

In [4]:
select_columns = ['lpep_pickup_datetime', 
                  'lpep_dropoff_datetime',
                  'PULocationID', 
                  'DOLocationID',
                  'passenger_count', 
                  'trip_distance', 
                  'tip_amount', 
                  'total_amount'
                  ]

df_green = pd.read_parquet(
    url, 
    columns= select_columns
    )

df_green.head(2)

,lpep_pickup_datetime,lpep_dropoff_datetime,PULocationID,DOLocationID,passenger_count,trip_distance,tip_amount,total_amount
0,2025-10-01 00:21:47,2025-10-01 00:24:37,247,69,1.0,0.70,1.70,10.00
1,2025-10-01 00:14:03,2025-10-01 00:24:14,66,25,1.0,1.61,2.78,16.68


In [5]:
# Handling missing values
df_green = df_green.replace(np.nan, 0)

In [6]:
df_green.info()

<class 'pandas.DataFrame'>
RangeIndex: 49416 entries, 0 to 49415
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   lpep_pickup_datetime   49416 non-null  datetime64[us]
 1   lpep_dropoff_datetime  49416 non-null  datetime64[us]
 2   PULocationID           49416 non-null  int32         
 3   DOLocationID           49416 non-null  int32         
 4   passenger_count        49416 non-null  float64       
 5   trip_distance          49416 non-null  float64       
 6   tip_amount             49416 non-null  float64       
 7   total_amount           49416 non-null  float64       
dtypes: datetime64[us](2), float64(4), int32(2)
memory usage: 2.6 MB


In [7]:
row = df_green.iloc[0, :]
row

lpep_pickup_datetime     2025-10-01 00:21:47
lpep_dropoff_datetime    2025-10-01 00:24:37
PULocationID                             247
DOLocationID                              69
passenger_count                          1.0
trip_distance                            0.7
tip_amount                               1.7
total_amount                            10.0
Name: 0, dtype: object

In [8]:
@dataclass   
class ride: 
    lpep_pickup_datetime: str
    lpep_dropoff_datetime: str
    PULocationID: int
    DOLocationID: int
    passenger_count: int 
    trip_distance: float
    tip_amount: float
    total_amount: float


In [9]:
def df_row_to_ride(row):
    return ride(
        lpep_pickup_datetime= str(row.lpep_pickup_datetime),
        lpep_dropoff_datetime= str(row.lpep_dropoff_datetime),
        PULocationID= int(row.PULocationID),
        DOLocationID= int(row.DOLocationID),
        passenger_count= int(row.passenger_count),
        trip_distance= float(row.trip_distance),
        tip_amount= float(row.tip_amount),
        total_amount= float(row.total_amount)
    )


In [10]:
trip = df_row_to_ride(df_green.iloc[0, :])
trip

ride(lpep_pickup_datetime='2025-10-01 00:21:47', lpep_dropoff_datetime='2025-10-01 00:24:37', PULocationID=247, DOLocationID=69, passenger_count=1, trip_distance=0.7, tip_amount=1.7, total_amount=10.0)

In [11]:
def ride_serializer(ride):
    ride_dict = dataclasses.asdict(ride)
    json_str = json.dumps(ride_dict)
    return json_str.encode('utf-8')

server = 'localhost:9092'

producer = KafkaProducer(
    bootstrap_servers= [server],
    value_serializer= ride_serializer
)

In [12]:
import time

topic_name= 'green-trips'
t0 = time.time()

for _, row in df_green.iterrows():
    trip = df_row_to_ride(row)
    producer.send(topic_name, value=trip)
    #print(f"Sent: {trip}")
    time.sleep(0.01)

producer.flush()

t1 = time.time()
print(f'took {(t1 - t0):.2f} seconds')

took 547.82 seconds
